---
title: Variational Autoencoders to Simulate GPS Signal
subtitle: Using the Geo-Life dataset to train an autoencoder and create more data
author: Sebastiano Ferraris
date: 2024-01-20
categories: [tutorial, geospatial, machine learning]
image: images/cover.png
toc: true
draft: false
twitter-card:
  image: images/cover.png
---


![](images/earth_models.png)


## Introduction

Draft!

This blog post is for the reader interested in building an intuition for how distances on the sphere are computed (@sec-map-projection, @sec-two-points-distance), to understand the details of the maths behind the Haversine distance (@sec-haversine-distance), to have an implementation in python with some examples and details about the numerical stability (@sec-implementation, @sec-numerical-stability), and a conclusion about what to use in the python practice (@sec-vincentry-formula). This last one is the only section you should look at if you are here to compute the Haversine distance quickly and accurately. 

The article starts with the somehow pedantic though possibly important section about the motivations and formulas behind the spherical model.


## Earth's Spherical Model {#sec-spherical-model}

In any dimension, the most symmetric geometrical object is the sphere. Unlike planes, spheres are also symmetric when considered within their embedding in an higher dimensional space. 

Even though the earth is not a perfect sphere, also for its symmetric properties, the 2D sphere embedded in the 3D space is a very reasonable earth's approximation. Amongst the models that are more accurate we can find the ellipsoid (which is the model used by the GPS systems, more precisely the [WGS84 geodetic system](https://en.wikipedia.org/wiki/World_Geodetic_System)), the [geoid model](https://www.sciencedirect.com/topics/earth-and-planetary-sciences/geoids), or even the local [topographic elevations](https://geopard.tech/blog/topography-and-relief-analytics-for-agricultural-fields).

Which model is the best one?

If the local topography is not relevant and the computations has to be kept bare simple and fast, the sphere is the go to model of the earth. The conventional coordinates system, consisting of the pair longitude (East-West direction) and latitude (North-South direction) measured in degrees from the center of the modelling surface in the 3D space, can be projected in any of the mentioned model, and even more, such as the cylindrical, conical and plane model. 

Adding the radius as third coordinate, we can model the altitude for each point. Unlike latitude and longitude, the location of the zero for the altitude point is model-dependent, and will be different if on the sphere, the ellipsoid or others. The functions mapping the latitude, longitude (and altitude) coordinates across models are called [maps projections](https://en.wikipedia.org/wiki/Map_projection)[^1].

For a fixed radius $R$, and for $\text{rad}: \text{Deg} \rightarrow \text{Rad}$ the function mapping degrees to radians, we can rename the two angles with the radians with the conventional greek letters $\theta$ (theta) and $\varphi$ (phi):

$$
\begin{align*}
\theta &:= \text{rad}(\text{Lon}) \\
\varphi &:= \text{rad}(\text{Lat}) \\
\end{align*}
$$ 

Another basic map for the spherical model is $\Pi: \mathbb{R}^2 \rightarrow \mathbb{S}^2 \subset \mathbb{R}^3$ that projects each pair $(\theta, \varphi)$ for $\theta \in [-\pi, \pi]$ and $\varphi \in [-\pi/2, \pi/2]$ on the sphere of radius $R$:

$$
\Pi(\theta, \varphi) = \begin{cases}
       x = R \cos(\varphi) \cos(\theta)\\
       y = R \cos(\varphi) \sin(\theta)\\
       z = R \sin(\varphi)\\
     \end{cases}
$$ 

The function $\Pi$ is one of the many map projections, and the only one we will consider in this blog post; the reasoning behind its formulation can be derived from the definition of sine and cosine and from the drawing below.

![](images/spherical_coordinates.png){#fig-spherical-coordinates}

Please note that most mathematical textbooks have the angle $\varphi$ at zero when the point in on the z-axis, so $\sin$ and $\cos$ are swapped, and the angle's domain would be $[0,\pi]$. In this blot post and to remain faithful to the definition of latitude, we consider $\varphi$ at zero when the point is on the xy-plane with domain $[-\pi,\pi]$. 


<!-- Footnote -->

[^1]: See for example by JP Snyder · 1987 ["Map projections: A working manual"](https://pubs.usgs.gov/publication/pp1395), U.S. Government Printing Office, or H.S. Roblin "Map projections", Edward Arnold Publisher Ltd.
